In [1]:
from IPython.display import display
import ast
import numpy as np
from typing import Sequence, Dict, Any, Optional
import pandas as pd
from scipy import stats

import main_analysis as ma

RESULTS_DIR = 'results'

pixdim[1,2,3] should be non-zero; setting 0 dims to 1


In [76]:
core_frois = ['FFA', 'EBA', 'PPA']
extra_frois = ['OFA', 'FBA', 'OPA']
extra_frois2 = ['VWFA', 'OVWFA']
all_extra_frois = extra_frois + extra_frois2
all_frois = core_frois + extra_frois + ['VWFA', 'OVWFA']

to_model_names = ma.get_model_names('task_optimized')
random_model_names = ma.get_model_names('untrained')
print(f'Found {len(to_model_names)} task-optimized models and {len(random_model_names)} random models.')
standard_model_name = 'resnet50imagenet1kv2'
best_model_name = 'vitb16siglip2res224'
examplerandom_model_name = 'alexnetrandom5'

Found 35 task-optimized models and 20 random models.


# Helper functions

In [3]:
def signflip_perm_test_against_ceiling(
    ceiling: Sequence[float],
    model: Sequence[float],
    *,
    alternative: str = "less",
    statistic: str = "mean",
    exact: bool = True,
    n_permutations: int = 100_000,
    seed: Optional[int] = 0,
    add_one_correction: bool = True,
) -> Dict[str, Any]:
    """
    Exact (or Monte Carlo) paired sign-flip permutation test for whether a model
    matches (or exceeds) a subject-specific ceiling.

    Inputs are length-8 (or length-N) sequences:
      ceiling[i] = subject i ceiling (e.g., median/mean of 7 brain-brain NC correlations)
      model[i]   = model-to-subject i noise-corrected correlation

    We test on paired differences d_i = model_i - ceiling_i.

    Parameters
    ----------
    alternative : {"less", "greater", "two-sided"}
        "less" tests whether model < ceiling (i.e., mean difference < 0).
    statistic : {"mean", "median"}
        Test statistic computed over d_i.
    exact : bool
        If True and N <= 20, enumerates all 2^N sign flips (exact p-value).
        If False, uses Monte Carlo sampling with n_permutations.
    n_permutations : int
        Number of random sign-flips if exact=False (ignored if exact=True).
    seed : int | None
        RNG seed for Monte Carlo (ignored if exact=True).
    add_one_correction : bool
        If True, uses (count+1)/(B+1) to avoid p=0 for finite permutations.

    Returns
    -------
    dict with keys:
      - p_value
      - t_obs
      - diffs (np.ndarray)
      - n (int)
      - n_perm (int)
      - alternative
      - statistic
    """
    ceiling = np.asarray(ceiling, dtype=float)
    model = np.asarray(model, dtype=float)

    if ceiling.shape != model.shape:
        raise ValueError(f"ceiling and model must have same shape; got {ceiling.shape} vs {model.shape}")
    if ceiling.ndim != 1:
        raise ValueError("ceiling and model must be 1D sequences")
    n = ceiling.size
    if n < 2:
        raise ValueError("Need at least 2 paired observations")

    diffs = model - ceiling  # d_i

    if statistic == "mean":
        stat_fn = np.mean
    elif statistic == "median":
        stat_fn = np.median
    else:
        raise ValueError("statistic must be 'mean' or 'median'")

    t_obs = float(stat_fn(diffs))

    # Build permutation distribution of T = stat_fn(s_i * diffs), where s_i in {-1, +1}
    if exact:
        if n > 20:
            raise ValueError("Exact enumeration is exponential; set exact=False for n > 20")
        # Enumerate all 2^n sign patterns via bit tricks
        B = 1 << n
        # signs: shape (B, n), entries in {-1, +1}
        bits = np.arange(B, dtype=np.uint32)[:, None]
        signs = 1.0 - 2.0 * ((bits >> np.arange(n, dtype=np.uint32)) & 1).astype(float)
        t_perm = stat_fn(signs * diffs[None, :], axis=1)
    else:
        rng = np.random.default_rng(seed)
        B = int(n_permutations)
        signs = rng.choice([-1.0, 1.0], size=(B, n), replace=True)
        t_perm = stat_fn(signs * diffs[None, :], axis=1)

    # Compute p-value
    if alternative == "less":
        count = np.sum(t_perm <= t_obs)
    elif alternative == "greater":
        count = np.sum(t_perm >= t_obs)
    elif alternative == "two-sided":
        count = np.sum(np.abs(t_perm) >= abs(t_obs))
    else:
        raise ValueError("alternative must be 'less', 'greater', or 'two-sided'")

    if add_one_correction:
        p_value = (count + 1.0) / (B + 1.0)
    else:
        p_value = count / float(B)

    return {
        "p_value": float(p_value),
        "t_obs": float(t_obs),
        "diffs": diffs,
        "n": int(n),
        "n_perm": int(B),
        "alternative": alternative,
        "statistic": statistic,
    }

# Section 2.1

In [ ]:
LOCALIZATION_VALIDATION_CSV_PATH = f'{RESULTS_DIR}/localization_validation_grillspector_val_prince2024.csv'

tthreshold = 7
random_tthreshold = 3

model_localizer = 'grillspector'

Found 35 task-optimized models and 20 random models.


In [4]:
locval_df = pd.read_csv(LOCALIZATION_VALIDATION_CSV_PATH)

standard_model_rows = locval_df[
    (locval_df['model_name'] == standard_model_name) &
    (locval_df['model_localization_localizer'] == model_localizer) &
    (locval_df['model_froi_tthreshold'] == tthreshold) &
    (locval_df['model_froi'].isin(core_frois))
]
standard_model_u_pvals = []
for _, srow in standard_model_rows.iterrows():
    row_u_pvals = ast.literal_eval(srow['validation_u_pvalues'])
    standard_model_u_pvals += row_u_pvals
print(f'Standard model {standard_model_name} has maximum p-value {np.max(standard_model_u_pvals)}')

best_model_rows = locval_df[
    (locval_df['model_name'] == best_model_name) &
    (locval_df['model_localization_localizer'] == model_localizer) &
    (locval_df['model_froi_tthreshold'] == tthreshold) &
    (locval_df['model_froi'].isin(core_frois))
]
best_model_u_pvals = []
for _, brow in best_model_rows.iterrows():
    row_u_pvals = ast.literal_eval(brow['validation_u_pvalues'])
    best_model_u_pvals += row_u_pvals
print(f'Best model {best_model_name} has maximum p-value {np.max(best_model_u_pvals)}')

to_model_rows = locval_df[
    (locval_df['model_name'].isin(to_model_names)) &
    (locval_df['model_localization_localizer'] == model_localizer) &
    (locval_df['model_froi_tthreshold'] == tthreshold) &
    (locval_df['model_froi'].isin(core_frois))
]
to_model_u_pvals = []
for _, trow in to_model_rows.iterrows():
    row_u_pvals = ast.literal_eval(trow['validation_u_pvalues'])
    to_model_u_pvals += row_u_pvals
print(f'Task-optimized models have maximum p-value {np.max(to_model_u_pvals)}')

print('\n\n')
examplerandom_model_rows = locval_df[
    (locval_df['model_name'] == examplerandom_model_name) &
    (locval_df['model_localization_localizer'] == model_localizer) &
    (locval_df['model_froi_tthreshold'] == random_tthreshold) &
    (locval_df['model_froi'].isin(core_frois))
]
examplerandom_model_u_pvals = []
for _, errow in examplerandom_model_rows.iterrows():
    row_u_pvals = ast.literal_eval(errow['validation_u_pvalues'])
    print(f'froi {errow["model_froi"]} p-values: {row_u_pvals}')

print('\n\n')
untrained_model_rows = locval_df[
    (locval_df['model_name'].isin(random_model_names)) &
    (locval_df['model_localization_localizer'] == model_localizer) &
    (locval_df['model_froi_tthreshold'] == random_tthreshold) &
    (locval_df['model_froi'].isin(core_frois))
]
untrained_model_u_pvals = []
n_untrained_frois = {froi: 0 for froi in core_frois}
for _, urrow in untrained_model_rows.iterrows():
    row_u_pvals = ast.literal_eval(urrow['validation_u_pvalues'])
    all_pval_lt_0pt05 = all([pval < 0.05 for pval in row_u_pvals])
    if all_pval_lt_0pt05:
        n_untrained_frois[urrow['model_froi']] += 1
        print(f'Untrained model {urrow["model_name"]} has p<0.05 in all runs for fROI {urrow["model_froi"]}: {row_u_pvals}')
    untrained_model_u_pvals += row_u_pvals
    # print(row_u_pvals)

print('\n\n')
for _, urrow in untrained_model_rows.iterrows():
    row_u_pvals = ast.literal_eval(urrow['validation_u_pvalues'])
    if urrow["model_name"] == examplerandom_model_name:
        print(f'Example random model {urrow["model_name"]} fROI {urrow["model_froi"]} p-values: {row_u_pvals}')

print('\n\n')
print(f'Untrained models have maximum p-value {np.max(untrained_model_u_pvals)}')
print(f'Number of untrained models with p<0.05 in all fROIs: {n_untrained_frois}')

Standard model resnet50imagenet1kv2 has maximum p-value 6.875277155242618e-19
Best model vitb16siglip2res224 has maximum p-value 1.0046503105672565e-22
Task-optimized models have maximum p-value 0.028550648235675383



froi EBA p-values: [1.8566716796386583e-07, 6.518249224950052e-21, 3.106691374822264e-15]
froi FFA p-values: [0.7979680061661435, 4.663211238633869e-10, 2.139183884285796e-08]
froi PPA p-values: [0.44099425935491926, 4.548307034099951e-12, 0.6694958320262913]



Untrained model alexnetrandom1 has p<0.05 in all runs for fROI PPA: [0.0072790559612759195, 1.4857634787857233e-11, 8.432285519074714e-05]
Untrained model alexnetrandom1 has p<0.05 in all runs for fROI EBA: [0.04741255354799484, 2.87031675133382e-24, 2.8296226491843953e-16]
Untrained model alexnetrandom2 has p<0.05 in all runs for fROI EBA: [3.244822427529779e-06, 3.1025332840738693e-20, 5.975445802834818e-14]
Untrained model alexnetrandom2 has p<0.05 in all runs for fROI PPA: [1.1528265957795565e-14, 2.221008976

# Section 2.2

In [4]:
SUBJECTSUBJECT_CORRELATIONS_CSV_PATH = f'{RESULTS_DIR}/subjectsubject_correlations_nsd515.csv'
MODELSUBJECT_CORRELATIONS_CSV_PATH = f'{RESULTS_DIR}/modelsubject_correlations_nsd515.csv'
FROIFROI_CORRELATIONS_CSV_PATH = f'{RESULTS_DIR}/froifroi_correlations_nsd515.csv'

tthreshold = 7

model_localizer = 'grillspector'

metrics = ['normed_pearsonr', 'normed_rdmr']
subjects = list(range(1, 9))

## Human-Human

In [5]:
hh_csv = pd.read_csv(SUBJECTSUBJECT_CORRELATIONS_CSV_PATH)
hh_csv_subset = hh_csv[
    (hh_csv['fmri_froi'].isin(core_frois)) &
    (hh_csv['fmri_tthreshold'] == tthreshold)
]

print(f'Minimum normed pearsonr across core ROIs and across all subject pairs:')
print(hh_csv_subset['normed_pearsonr'].min().item())
print(f'Maximum normed pearsonr p-value across core ROIs and across all subject pairs:')
print(hh_csv_subset['pvalue_pearsonr'].max().item())
print(f'Minimum normed rdmr across core ROIs and across all subject pairs:')
print(hh_csv_subset['normed_rdmr'].min().item())
print(f'Maximum normed rdmr p-value across core ROIs and across all subject pairs:')
print(hh_csv_subset['pvalue_rdmr'].max().item())


hh_corrs_csv_records = []
median_correlations_records = []
for froi in core_frois:
    record = {'fmri_froi': froi}
    hh_median_correlations = {metric: [] for metric in metrics}
    for subject in subjects:
        hh_record = {'fmri_froi': froi, 'subject': subject}
        hh_csv_ = hh_csv[
            (hh_csv['fmri_froi'] == froi) &
            (hh_csv['fmri_tthreshold'] == tthreshold) &
            (
                (hh_csv['subject1'] == subject) | 
                (hh_csv['subject2'] == subject)
            )
        ]
        assert len(hh_csv_) == 7, f'Expected 7 rows for subject {subject} and froi {froi}, but got {len(hh_csv_)}'
        for metric in metrics:
            median_metric = hh_csv_[metric].median()
            hh_record[metric] = median_metric
            hh_median_correlations[metric].append(median_metric)
        hh_corrs_csv_records.append(hh_record)
    for metric in metrics:
        record[f'{metric}_median'] = np.median(hh_median_correlations[metric])
    median_correlations_records.append(record)
hh_corrs_csv = pd.DataFrame(hh_corrs_csv_records)
hh_median_csv = pd.DataFrame(median_correlations_records)
# hh_csv_grouped = hh_csv.groupby(['fmri_froi']).aggregate(
#     {
#         'normed_pearsonr': ['median'],
#         'normed_rdmr': ['median'],
#     }
# )
# hh_csv_grouped = (
#     hh_csv
#     .groupby(['fmri_froi'])
#     .agg(
#         normed_pearsonr_median=('normed_pearsonr', 'median'),
#         normed_rdmr_median=('normed_rdmr', 'median'),
#     )
#     .reset_index()
# ).set_index('fmri_froi')
display(hh_median_csv)
print('Median normed pearsonr across core ROIs:')
print(hh_median_csv['normed_pearsonr_median'].median().item())
print('Median normed rdmr across core ROIs:')
print(hh_median_csv['normed_rdmr_median'].median().item())

Minimum normed pearsonr across core ROIs and across all subject pairs:
0.8727959208707898
Maximum normed pearsonr p-value across core ROIs and across all subject pairs:
1.4775544305207639e-38
Minimum normed rdmr across core ROIs and across all subject pairs:
0.8050828196815112
Maximum normed rdmr p-value across core ROIs and across all subject pairs:
0.0


,fmri_froi,normed_pearsonr_median,normed_rdmr_median
0,FFA,0.981243,0.996540
1,PPA,1.005868,1.022174
2,EBA,0.991892,1.091029


Median normed pearsonr across core ROIs:
0.991892232542822
Median normed rdmr across core ROIs:
1.0221744599882632


## Model-Human

In [6]:
mh_csv = pd.read_csv(MODELSUBJECT_CORRELATIONS_CSV_PATH)
mh_csv = mh_csv[
    (mh_csv['fmri_froi'].isin(core_frois)) &
    (mh_csv['fmri_tthreshold'] == tthreshold) &
    (mh_csv['model_name'].isin(to_model_names)) &
    (mh_csv['model_localizer'] == model_localizer) &
    (mh_csv['model_froi'].isin(core_frois)) &
    (mh_csv['model_tthreshold'] == tthreshold)
]


mh_csv_grouped = (
    mh_csv
    .groupby(['fmri_froi', 'model_name'])
    .agg(
        normed_pearsonr_median=('normed_pearsonr', 'median'),
        normed_rdmr_median=('normed_rdmr', 'median'),
    )
    .reset_index()
)

# mh_csv_grouped_best = mh_csv_grouped[
#     mh_csv_grouped['model_name'] == best_model_name
# ]
# # drop the model_name column & set fmri_froi as index
# mh_csv_grouped_best = mh_csv_grouped_best.drop(columns=['model_name']).set_index('fmri_froi')
# print(f'Model-Subject correlations for best model: {best_model_name}')
# display(mh_csv_grouped_best)

mh_csv_grouped2 = (
    mh_csv_grouped
    .groupby('fmri_froi')
    .agg(
        normed_pearsonr_median=('normed_pearsonr_median', 'median'),
        normed_rdmr_median=('normed_rdmr_median', 'median'),
    )
    .reset_index()
).set_index('fmri_froi')
print('Model-Subject correlations (grouped by ROI):')
display(mh_csv_grouped2)

median_delta_records = []
for froi in core_frois:
    for model_name in to_model_names:
        record = {'fmri_froi': froi, 'model_name': model_name}
        model_froi_deltas = {metric: [] for metric in metrics}
        for subject in subjects:
            subject_froi_hh_corr = hh_corrs_csv[
                (hh_corrs_csv['fmri_froi'] == froi) &
                (hh_corrs_csv['subject'] == subject)
            ]
            model_froi_mh_corr = mh_csv[
                (mh_csv['model_name'] == model_name) & 
                (mh_csv['fmri_froi'] == froi) & 
                (mh_csv['model_froi'] == froi) & 
                (mh_csv['subject'] == subject) 
            ]
            for metric in metrics:
                model_froi_deltas[metric].append(
                    subject_froi_hh_corr[metric].values[0] - 
                    model_froi_mh_corr[metric].values[0]
                )
        for metric in metrics:
            record[f'{metric}_median'] = np.median(model_froi_deltas[metric])
        median_delta_records.append(record)
mh_deltas_median_csv = pd.DataFrame(median_delta_records)

print('Delta Rs between model-subject and subject-subject:')
display(mh_deltas_median_csv)

mh_deltas_median_csv_grouped = (
    mh_deltas_median_csv
    .groupby('fmri_froi')
    .agg(
        normed_pearsonr_median=('normed_pearsonr_median', 'median'),
        normed_rdmr_median=('normed_rdmr_median', 'median'),
    )
    .reset_index()
).set_index('fmri_froi')

print('Delta Rs between model-subject and subject-subject, medians:')
# delta Rs between model-subject and subject-subject medians
median_delta_df = hh_median_csv.set_index('fmri_froi') - mh_csv_grouped2
display(mh_deltas_median_csv_grouped)

print('Delta Rs between best model-subject and subject-subject medians:')
# delta Rs between best model-subject and subject-subject medians
display(mh_deltas_median_csv[mh_deltas_median_csv['model_name'] == best_model_name])

# print(f'One sided U-test between subject-subject and all model-subject correlations:')
# # u-test between subject-subject and model-subject correlations, for each metric and fROI
# for froi in core_frois:
#     print(f'\nFROI: {froi}')
#     for metric in metrics:
#         hh_values = hh_csv[
#             (hh_csv['fmri_froi'] == froi)
#         ][metric].values
#         mh_values = mh_csv[
#             (mh_csv['fmri_froi'] == froi)
#         ][metric].values
#         u_stat, p_value = stats.mannwhitneyu(hh_values, mh_values, alternative='greater')
#         print(f'Metric: {metric}, U-statistic: {u_stat}, p-value: {p_value},    median subject-subject: {np.median(hh_values)}, median model-subject: {np.median(mh_values)}')

Model-Subject correlations (grouped by ROI):


,normed_pearsonr_median,normed_rdmr_median
fmri_froi,,
EBA,0.672800,0.306611
FFA,0.618296,0.383438
PPA,0.726980,0.367889


Delta Rs between model-subject and subject-subject:


,fmri_froi,model_name,normed_pearsonr_median,normed_rdmr_median
0,FFA,alexnetimagenet1kv1,0.393286,0.586167
1,FFA,alexnetipclref01,0.539089,0.741135
2,FFA,alexnetbarlowtwins,0.434835,0.704224
3,FFA,vgg16imagenet1kv1,0.279228,0.540651
4,FFA,resnet18imagenet1kv1,0.422929,0.689270
...,...,...,...,...
100,EBA,vitb16synclr,0.297792,0.778061
101,EBA,vitb14dinov2,0.324810,0.906172
102,EBA,vitb14dinov2reg,0.277168,0.882603
103,EBA,vitl14openaiclip,0.347178,0.809444


Delta Rs between model-subject and subject-subject, medians:


,normed_pearsonr_median,normed_rdmr_median
fmri_froi,,
EBA,0.332605,0.782361
FFA,0.361975,0.635731
PPA,0.280907,0.681929


Delta Rs between best model-subject and subject-subject medians:


,fmri_froi,model_name,normed_pearsonr_median,normed_rdmr_median
28,FFA,vitb16siglip2res224,0.321903,0.430301
63,PPA,vitb16siglip2res224,0.305676,0.750250
98,EBA,vitb16siglip2res224,0.247332,0.618488


### ANN-brain < brain-brain corrs

#### Individual model level

In [8]:
hh_lt_mh_stats_records = []
for metric in metrics:
    print(f'\n\nPaired sign-flip permutation test for whether model-subject {metric} is less than subject-subject {metric} (i.e., model is worse than ceiling):')
    for froi in core_frois:
        for model_name in to_model_names:
            hh_ceilings = []
            mh_corrs = []
            record = {'fmri_froi': froi, 'model_name': model_name, 'metric': metric}
            for subject in subjects:
                hh_ceiling = hh_corrs_csv[
                    (hh_corrs_csv['fmri_froi'] == froi) & 
                    (hh_corrs_csv['subject'] == subject)
                ][metric].values[0]
                hh_ceilings.append(hh_ceiling)
                mh_corr = mh_csv[
                    (mh_csv['model_name'] == model_name) &
                    (mh_csv['fmri_froi'] == froi) &
                    (mh_csv['subject'] == subject)
                ][metric].values[0]
                mh_corrs.append(mh_corr)
            test_result = signflip_perm_test_against_ceiling(
                ceiling=hh_ceilings,
                model=mh_corrs,
                alternative='less',  # test model < ceiling
                statistic='median',
                exact=True, # N=8 -> exact 256 perms
                n_permutations=100_000,
                seed=0,
                add_one_correction=True,
            )
            record.update(test_result)
            # print(f'FROI: {froi}, model: {model_name}, metric: {metric}, p-value: {test_result["p_value"]}, median difference: {test_result["t_obs"]}, median subject-subject: {np.median(hh_ceilings)}, median model-subject: {np.median(mh_corrs)}')
            hh_lt_mh_stats_records.append(record)
hh_lt_mh_stats_df = pd.DataFrame(hh_lt_mh_stats_records)
print('Paired sign-flip permutation test results for whether model-subject correlations are less than subject-subject ceilings:')
display(hh_lt_mh_stats_df)
print('Number of models that are significantly worse than ceiling (p<0.05):')
print(f"{hh_lt_mh_stats_df[hh_lt_mh_stats_df['p_value'] < 0.05].shape[0]} out of {hh_lt_mh_stats_df.shape[0]} tests")



Paired sign-flip permutation test for whether model-subject normed_pearsonr is less than subject-subject normed_pearsonr (i.e., model is worse than ceiling):


Paired sign-flip permutation test for whether model-subject normed_rdmr is less than subject-subject normed_rdmr (i.e., model is worse than ceiling):
Paired sign-flip permutation test results for whether model-subject correlations are less than subject-subject ceilings:


,fmri_froi,model_name,metric,p_value,t_obs,diffs,n,n_perm,alternative,statistic
0,FFA,alexnetimagenet1kv1,normed_pearsonr,0.035019,-0.393286,"[-0.3971186042285062, -0.28034553270052, -0.38...",8,256,less,median
1,FFA,alexnetipclref01,normed_pearsonr,0.035019,-0.539089,"[-0.54987765191179, -0.44754587509776117, -0.5...",8,256,less,median
2,FFA,alexnetbarlowtwins,normed_pearsonr,0.035019,-0.434835,"[-0.39339922110661296, -0.3491208306732996, -0...",8,256,less,median
3,FFA,vgg16imagenet1kv1,normed_pearsonr,0.035019,-0.279228,"[-0.2745375913460769, -0.22137872322426833, -0...",8,256,less,median
4,FFA,resnet18imagenet1kv1,normed_pearsonr,0.035019,-0.422929,"[-0.4126482974744229, -0.41845951979970053, -0...",8,256,less,median
...,...,...,...,...,...,...,...,...,...,...
205,EBA,vitb16synclr,normed_rdmr,0.035019,-0.778061,"[-0.7614909801872976, -0.7998049330526149, -0....",8,256,less,median
206,EBA,vitb14dinov2,normed_rdmr,0.035019,-0.906172,"[-0.8799971816915746, -0.9150429688198813, -0....",8,256,less,median
207,EBA,vitb14dinov2reg,normed_rdmr,0.035019,-0.882603,"[-0.8699307544675581, -0.9094887363933883, -0....",8,256,less,median
208,EBA,vitl14openaiclip,normed_rdmr,0.035019,-0.809444,"[-0.8066846689238009, -0.836458378729446, -0.8...",8,256,less,median


Number of models that are significantly worse than ceiling (p<0.05):
210 out of 210 tests


#### Population level

In [14]:
hh_lt_mh_stats_df_pop = hh_lt_mh_stats_df.copy()
# Add a new column for median_diffs
hh_lt_mh_stats_df_pop['median_diff'] = hh_lt_mh_stats_df_pop['diffs'].apply(np.median)
print('Running a population-level test of whether model correlations are less than ceiling correlations, across all models and ROIs')
print('Wilcoxon signed-rank test on median differences across all models and ROIs, againts the null hypothesis that the median difference is zero (i.e., model = ceiling):')
for froi in core_frois:
    print('\nFROI:', froi)
    for metric in metrics:
        
        median_diffs = hh_lt_mh_stats_df_pop[
            (hh_lt_mh_stats_df_pop['fmri_froi'] == froi) &
            (hh_lt_mh_stats_df_pop['metric'] == metric)
        ]['median_diff'].values
        stat, p_value = stats.wilcoxon(median_diffs, alternative='less', zero_method='wilcox')
        print(f'FROI: {froi}, metric: {metric}, Wilcoxon stat: {stat}, p-value: {p_value}, median of median differences: {np.median(median_diffs)}')

Running a population-level test of whether model correlations are less than ceiling correlations, across all models and ROIs
Wilcoxon signed-rank test on median differences across all models and ROIs, againts the null hypothesis that the median difference is zero (i.e., model = ceiling):

FROI: FFA
FROI: FFA, metric: normed_pearsonr, Wilcoxon stat: 0.0, p-value: 2.9103830456733704e-11, median of median differences: -0.3619754872275579
FROI: FFA, metric: normed_rdmr, Wilcoxon stat: 0.0, p-value: 2.9103830456733704e-11, median of median differences: -0.6357314835238632

FROI: PPA
FROI: PPA, metric: normed_pearsonr, Wilcoxon stat: 0.0, p-value: 2.9103830456733704e-11, median of median differences: -0.2809072722460479
FROI: PPA, metric: normed_rdmr, Wilcoxon stat: 0.0, p-value: 2.9103830456733704e-11, median of median differences: -0.6819286929375619

FROI: EBA
FROI: EBA, metric: normed_pearsonr, Wilcoxon stat: 0.0, p-value: 2.9103830456733704e-11, median of median differences: -0.33260546

#### Legacy

In [8]:
pvalues_dict = {}
model_medians_dict = {}
for metric in metrics:
    metric_min_pval = np.inf
    pvalues_dict[metric] = {}
    model_medians_dict[metric] = {}
    print(f'For metric: {metric}')
    for froi in core_frois:
        pvalues_dict[metric][froi] = {}
        model_medians_dict[metric][froi] = {}
        print(f'  For ROI: {froi}')
        hh_subset = hh_csv[
            (hh_csv['fmri_froi'] == froi)
        ]
        hh_corrs = hh_subset[metric].values
        for model_name in to_model_names:
            subset = mh_csv[
                (mh_csv['fmri_froi'] == froi) &
                (mh_csv['model_name'] == model_name)
            ]
            mh_corrs = subset[metric].values
            u_stat, p_value = stats.mannwhitneyu(
                mh_corrs,
                hh_corrs,
                alternative='less'
            )
            if p_value >= 0.05:
                print(f'    Model {model_name} p-value: {p_value} (NOT SIGNIFICANT)')
            # else:
            #     print(f'    Model {model_name} p-value: {p_value} (SIGNIFICANT)')
            pvalues_dict[metric][froi][model_name] = p_value
            model_medians_dict[metric][froi][model_name] = np.median(mh_corrs)
            if p_value < metric_min_pval:
                metric_min_pval = p_value
        metric_froi_best_model_median = np.median(model_medians_dict[metric][froi][best_model_name])
        metric_froi_model_medians = model_medians_dict[metric][froi].values()
        median_of_medians = np.median(list(metric_froi_model_medians))
        print(f'    Median of model medians for ROI {froi} and metric {metric}: {median_of_medians}')
        # Print the median for the best model
        print(f'    Median of {best_model_name} for ROI {froi} and metric {metric}: {metric_froi_best_model_median}')
    print('\n')
    print(f'Minimum p-value for metric {metric} across core ROIs and models: {metric_min_pval}')
    print('\n')

For metric: normed_pearsonr
  For ROI: FFA
    Median of model medians for ROI FFA and metric normed_pearsonr: 0.6182962524942974
    Median of vitb16siglip2res224 for ROI FFA and metric normed_pearsonr: 0.6577928973056599
  For ROI: PPA
    Median of model medians for ROI PPA and metric normed_pearsonr: 0.7269798643138887
    Median of vitb16siglip2res224 for ROI PPA and metric normed_pearsonr: 0.7118005729011971
  For ROI: EBA
    Median of model medians for ROI EBA and metric normed_pearsonr: 0.6727995765455339
    Median of vitb16siglip2res224 for ROI EBA and metric normed_pearsonr: 0.7641392044521946


Minimum p-value for metric normed_pearsonr across core ROIs and models: 3.3046555326212464e-08


For metric: normed_rdmr
  For ROI: FFA
    Median of model medians for ROI FFA and metric normed_rdmr: 0.3834384870617388
    Median of vitb16siglip2res224 for ROI FFA and metric normed_rdmr: 0.5741843944220171
  For ROI: PPA
    Median of model medians for ROI PPA and metric normed_rdmr

In [30]:
# Subtract the grouped model-subject correlations from the grouped subject-subject correlations
print('GAP between subject-subject and model-subject correlations (grouped by ROI):')
median_gap_df = hh_csv_grouped - mh_csv_grouped2
display(median_gap_df)

print(f'Model-Subject correlations for best model: {best_model_name}')
display(mh_csv_grouped_best)

print(f'GAP between subject-subject and model-subject correlations for the best model {best_model_name} (grouped by ROI):')
best_gap_df = hh_csv_grouped - mh_csv_grouped_best
display(best_gap_df)

GAP between subject-subject and model-subject correlations (grouped by ROI):


,normed_pearsonr_median,normed_rdmr_median
fmri_froi,,
EBA,0.324837,0.820833
FFA,0.365212,0.626231
PPA,0.287494,0.701125


Model-Subject correlations for best model: vitb16siglip2res224


,normed_pearsonr_median,normed_rdmr_median
fmri_froi,,
EBA,0.764139,0.481560
FFA,0.657793,0.574184
PPA,0.711801,0.310059


GAP between subject-subject and model-subject correlations for the best model vitb16siglip2res224 (grouped by ROI):


,normed_pearsonr_median,normed_rdmr_median
fmri_froi,,
EBA,0.233497,0.645884
FFA,0.325715,0.435485
PPA,0.302673,0.758955


## fROI-fROI

In [39]:
froifroi_df = pd.read_csv(FROIFROI_CORRELATIONS_CSV_PATH)

In [61]:
considered_frois = core_frois + extra_frois

median_froifroi_correlations_records = []
for froi1 in considered_frois:
    for froi2 in considered_frois:
        froi1_froi2_df = froifroi_df[
            (froifroi_df['fmri_froi1'] == froi1) &
            (froifroi_df['fmri_froi2'] == froi2)
        ]
        if len(froi1_froi2_df) == 0:
            continue
        froi1_froi2_median_correlations = {metric: [] for metric in metrics}
        for subject in subjects:
            record = {'fmri_froi1': froi1, 'fmri_froi2': froi2, 'subject': subject}
            froifroi_df_ = froifroi_df[
                (froifroi_df['fmri_froi1'] == froi1) &
                (froifroi_df['fmri_froi2'] == froi2) &
                (froifroi_df['fmri_tthreshold'] == tthreshold) &
                (froifroi_df['subject1'] == subject)
            ]
            assert len(froifroi_df_) == 7, f'Expected 7 rows for subject {subject} and froi {froi1}-{froi2}, but got {len(froifroi_df_)}'
            for metric in metrics:
                median_metric = froifroi_df_[metric].median()
                record[f'{metric}_median'] = median_metric
                # froi1_froi2_median_correlations[metric].append(median_metric)
        # for metric in metrics:
        #     record[f'{metric}_median'] = np.median(froi1_froi2_median_correlations[metric])
            median_froifroi_correlations_records.append(record)
froifroi_median_csv = pd.DataFrame(median_froifroi_correlations_records)


# froifroi_grouped = (
#     froifroi_df
#     .groupby(['fmri_froi1', 'fmri_froi2'])
#     .agg(
#         normed_pearsonr_median=('normed_pearsonr', 'median'),
#         normed_rdmr_median=('normed_rdmr', 'median'),
#     )
#     .reset_index()
# ).set_index(['fmri_froi1', 'fmri_froi2'])
# display(froifroi_grouped)

In [65]:
fmri_froi1 = 'FFA'
fmri_froi2s = ['OFA', 'EBA', 'PPA']
for metric in metrics:
    print(f'\n{metric}:')
    for fmri_froi2 in fmri_froi2s:
        f1f1_f1f2_deltas = []
        for subject in subjects:
            f1f1_corr = froifroi_median_csv[
                (froifroi_median_csv['fmri_froi1'] == fmri_froi1) & 
                (froifroi_median_csv['fmri_froi2'] == fmri_froi1) 
            ][f'{metric}_median'].values[0]
            f1f2_corr = froifroi_median_csv[
                (froifroi_median_csv['fmri_froi1'] == fmri_froi1) & 
                (froifroi_median_csv['fmri_froi2'] == fmri_froi2) 
            ][f'{metric}_median'].values[0]
            f1f1_f1f2_delta = f1f1_corr - f1f2_corr
            f1f1_f1f2_deltas.append(f1f1_f1f2_delta)
        assert len(f1f1_f1f2_deltas) == 8
        print(f'Median delta between {fmri_froi1} between {fmri_froi1} and {fmri_froi2} = {np.median(f1f1_f1f2_deltas)}')


normed_pearsonr:
Median delta between FFA between FFA and OFA = 0.062485856140535145
Median delta between FFA between FFA and EBA = 0.12320201234063766
Median delta between FFA between FFA and PPA = 1.610699027964154

normed_rdmr:
Median delta between FFA between FFA and OFA = 0.04846584956698352
Median delta between FFA between FFA and EBA = 0.19004486793110542
Median delta between FFA between FFA and PPA = 0.5788717073914761


# Section 2.4

## t-thresholds

In [79]:
tthresholds = [3, 5, 7, 9, 11]
model_localizer = 'grillspector'

SUBJECTSUBJECT_CORRELATIONS_CSV_PATH = f'{RESULTS_DIR}/subjectsubject_correlations_nsd515.csv'
MODELSUBJECT_CORRELATIONS_CSV_PATH = f'{RESULTS_DIR}/tthreshold_effect_correlations_nsd515.csv'

metrics = ['normed_pearsonr', 'normed_rdmr']
subjects = list(range(1, 9)) 

### Population-level

In [93]:
min_median_deltas = {froi: {metric: (np.inf, (None, None)) for metric in metrics} for froi in core_frois}
for fmri_tthreshold in tthresholds:
    for model_tthreshold in tthresholds:
        print(f'\n\nComparing subject-subject and model-subject correlations for fROI t-threshold {fmri_tthreshold} and model t-threshold {model_tthreshold}...')        
        hh_csv = pd.read_csv(SUBJECTSUBJECT_CORRELATIONS_CSV_PATH)
        hh_csv_subset = hh_csv[
            (hh_csv['fmri_froi'].isin(core_frois)) &
            (hh_csv['fmri_tthreshold'] == fmri_tthreshold)
        ]

        # print(f'Minimum normed pearsonr across core ROIs and across all subject pairs:')
        # print(hh_csv_subset['normed_pearsonr'].min().item())
        # print(f'Maximum normed pearsonr p-value across core ROIs and across all subject pairs:')
        # print(hh_csv_subset['pvalue_pearsonr'].max().item())
        # print(f'Minimum normed rdmr across core ROIs and across all subject pairs:')
        # print(hh_csv_subset['normed_rdmr'].min().item())
        # print(f'Maximum normed rdmr p-value across core ROIs and across all subject pairs:')
        # print(hh_csv_subset['pvalue_rdmr'].max().item())


        hh_corrs_csv_records = []
        median_correlations_records = []
        for froi in core_frois:
            record = {'fmri_froi': froi}
            hh_median_correlations = {metric: [] for metric in metrics}
            for subject in subjects:
                hh_record = {'fmri_froi': froi, 'subject': subject}
                hh_csv_ = hh_csv[
                    (hh_csv['fmri_froi'] == froi) &
                    (hh_csv['fmri_tthreshold'] == fmri_tthreshold) &
                    (
                        (hh_csv['subject1'] == subject) | 
                        (hh_csv['subject2'] == subject)
                    )
                ]
                assert len(hh_csv_) == 7, f'Expected 7 rows for subject {subject} and froi {froi}, but got {len(hh_csv_)}'
                for metric in metrics:
                    median_metric = hh_csv_[metric].median()
                    hh_record[metric] = median_metric
                    hh_median_correlations[metric].append(median_metric)
                hh_corrs_csv_records.append(hh_record)
            for metric in metrics:
                record[f'{metric}_median'] = np.median(hh_median_correlations[metric])
            median_correlations_records.append(record)
        hh_corrs_csv = pd.DataFrame(hh_corrs_csv_records)
        hh_median_csv = pd.DataFrame(median_correlations_records)

        mh_csv = pd.read_csv(MODELSUBJECT_CORRELATIONS_CSV_PATH)
        mh_csv = mh_csv[
            (mh_csv['fmri_froi'].isin(core_frois)) &
            (mh_csv['fmri_tthreshold'] == fmri_tthreshold) &
            (mh_csv['model_name'].isin(to_model_names)) &
            (mh_csv['model_localizer'] == model_localizer) &
            (mh_csv['model_froi'].isin(core_frois)) &
            (mh_csv['model_tthreshold'] == model_tthreshold)
        ]

        hh_lt_mh_stats_records = []
        for metric in metrics:
            # print(f'\n\nPaired sign-flip permutation test for whether model-subject {metric} is less than subject-subject {metric} (i.e., model is worse than ceiling):')
            for froi in core_frois:
                # print(f'\nFROI: {froi}')
                for model_name in to_model_names:
                    hh_ceilings = []
                    mh_corrs = []
                    record = {'fmri_froi': froi, 'model_name': model_name, 'metric': metric}
                    for subject in subjects:
                        hh_ceiling = hh_corrs_csv[
                            (hh_corrs_csv['fmri_froi'] == froi) & 
                            (hh_corrs_csv['subject'] == subject)
                        ][metric].values[0]
                        hh_ceilings.append(hh_ceiling)
                        mh_corr = mh_csv[
                            (mh_csv['model_name'] == model_name) &
                            (mh_csv['fmri_froi'] == froi) &
                            (mh_csv['model_froi'] == froi) &
                            (mh_csv['subject'] == subject)
                        ][metric].values[0]
                        mh_corrs.append(mh_corr)
                    test_result = signflip_perm_test_against_ceiling(
                        ceiling=hh_ceilings,
                        model=mh_corrs,
                        alternative='less',  # test model < ceiling
                        statistic='median',
                        exact=True, # N=8 -> exact 256 perms
                        n_permutations=100_000,
                        seed=0,
                        add_one_correction=True,
                    )
                    record.update(test_result)
                    # print(f'FROI: {froi}, model: {model_name}, metric: {metric}, p-value: {test_result["p_value"]}, median difference: {test_result["t_obs"]}, median subject-subject: {np.median(hh_ceilings)}, median model-subject: {np.median(mh_corrs)}')
                    hh_lt_mh_stats_records.append(record)
        hh_lt_mh_stats_df = pd.DataFrame(hh_lt_mh_stats_records)
        # print('Paired sign-flip permutation test results for whether model-subject correlations are less than subject-subject ceilings:')
        # display(hh_lt_mh_stats_df)
        # print('Number of models that are significantly worse than ceiling (p<0.05):')
        print(f"{hh_lt_mh_stats_df[hh_lt_mh_stats_df['p_value'] < 0.05].shape[0]} out of {hh_lt_mh_stats_df.shape[0]} tests")
        # assert len(hh_lt_mh_stats_df['p_value'].unique()) == 1, 'Expected all p-values to be the same across models and ROIs since we are using exact permutations with add-one correction, but got multiple unique p-values'

        hh_lt_mh_stats_df_pop = hh_lt_mh_stats_df.copy()
        # Add a new column for median_diffs
        hh_lt_mh_stats_df_pop['median_diff'] = hh_lt_mh_stats_df_pop['diffs'].apply(np.median)
        # print('Running a population-level test of whether model correlations are less than ceiling correlations, across all models and ROIs')
        # print('Wilcoxon signed-rank test on median differences across all models and ROIs, againts the null hypothesis that the median difference is zero (i.e., model = ceiling):')
        for froi in core_frois:
            print('\nFROI:', froi)
            for metric in metrics:
                best_model_median_diff = hh_lt_mh_stats_df_pop[
                    (hh_lt_mh_stats_df_pop['model_name'] == best_model_name) &
                    (hh_lt_mh_stats_df_pop['fmri_froi'] == froi) &
                    (hh_lt_mh_stats_df_pop['metric'] == metric)
                ]['median_diff'].values[0]
                median_diffs = hh_lt_mh_stats_df_pop[
                    (hh_lt_mh_stats_df_pop['fmri_froi'] == froi) &
                    (hh_lt_mh_stats_df_pop['metric'] == metric)
                ]['median_diff'].values
                stat, p_value = stats.wilcoxon(median_diffs, alternative='less', zero_method='wilcox')
                print(f'FROI: {froi}, metric: {metric}, Wilcoxon stat: {stat}, p-value: {p_value}, median of median differences: {np.median(median_diffs)}')
                if np.abs(best_model_median_diff) < min_median_deltas[froi][metric][0]:
                    min_median_deltas[froi][metric] = (np.abs(best_model_median_diff), (fmri_tthreshold, model_tthreshold))
    print('\n\n')
print('Minimum of median differences across all t-threshold combinations:')
for froi in core_frois:
    for metric in metrics:
        print(f'FROI: {froi}, Metric: {metric}, minimum median difference: {min_median_deltas[froi][metric]}')



Comparing subject-subject and model-subject correlations for fROI t-threshold 3 and model t-threshold 3...
210 out of 210 tests

FROI: FFA
FROI: FFA, metric: normed_pearsonr, Wilcoxon stat: 0.0, p-value: 2.9103830456733704e-11, median of median differences: -0.40784360298096783
FROI: FFA, metric: normed_rdmr, Wilcoxon stat: 0.0, p-value: 2.9103830456733704e-11, median of median differences: -0.7251100613424022

FROI: EBA
FROI: EBA, metric: normed_pearsonr, Wilcoxon stat: 0.0, p-value: 2.9103830456733704e-11, median of median differences: -0.38020688503070105
FROI: EBA, metric: normed_rdmr, Wilcoxon stat: 0.0, p-value: 2.9103830456733704e-11, median of median differences: -0.8506662003424978

FROI: PPA
FROI: PPA, metric: normed_pearsonr, Wilcoxon stat: 0.0, p-value: 2.9103830456733704e-11, median of median differences: -0.32614955372370813
FROI: PPA, metric: normed_rdmr, Wilcoxon stat: 0.0, p-value: 2.9103830456733704e-11, median of median differences: -0.7666958941781223


Comparing 

## Localizers

### Intersections

### Other localizer

In [64]:
tthreshold = 7
model_localizer = 'kanwisher'

SUBJECTSUBJECT_CORRELATIONS_CSV_PATH = f'{RESULTS_DIR}/subjectsubject_correlations_nsd515.csv'
MODELSUBJECT_CORRELATIONS_CSV_PATH = f'{RESULTS_DIR}/modelsubject_correlations_nsd515.csv'

metrics = ['normed_pearsonr', 'normed_rdmr']
subjects = list(range(1, 9)) 

In [65]:
hh_csv = pd.read_csv(SUBJECTSUBJECT_CORRELATIONS_CSV_PATH)
hh_csv_subset = hh_csv[
    (hh_csv['fmri_froi'].isin(core_frois)) &
    (hh_csv['fmri_tthreshold'] == tthreshold)
]

print(f'Minimum normed pearsonr across core ROIs and across all subject pairs:')
print(hh_csv_subset['normed_pearsonr'].min().item())
print(f'Maximum normed pearsonr p-value across core ROIs and across all subject pairs:')
print(hh_csv_subset['pvalue_pearsonr'].max().item())
print(f'Minimum normed rdmr across core ROIs and across all subject pairs:')
print(hh_csv_subset['normed_rdmr'].min().item())
print(f'Maximum normed rdmr p-value across core ROIs and across all subject pairs:')
print(hh_csv_subset['pvalue_rdmr'].max().item())


hh_corrs_csv_records = []
median_correlations_records = []
for froi in core_frois:
    record = {'fmri_froi': froi}
    hh_median_correlations = {metric: [] for metric in metrics}
    for subject in subjects:
        hh_record = {'fmri_froi': froi, 'subject': subject}
        hh_csv_ = hh_csv[
            (hh_csv['fmri_froi'] == froi) &
            (hh_csv['fmri_tthreshold'] == tthreshold) &
            (
                (hh_csv['subject1'] == subject) | 
                (hh_csv['subject2'] == subject)
            )
        ]
        assert len(hh_csv_) == 7, f'Expected 7 rows for subject {subject} and froi {froi}, but got {len(hh_csv_)}'
        for metric in metrics:
            median_metric = hh_csv_[metric].median()
            hh_record[metric] = median_metric
            hh_median_correlations[metric].append(median_metric)
        hh_corrs_csv_records.append(hh_record)
    for metric in metrics:
        record[f'{metric}_median'] = np.median(hh_median_correlations[metric])
    median_correlations_records.append(record)
hh_corrs_csv = pd.DataFrame(hh_corrs_csv_records)
hh_median_csv = pd.DataFrame(median_correlations_records)

Minimum normed pearsonr across core ROIs and across all subject pairs:
0.8727959208707898
Maximum normed pearsonr p-value across core ROIs and across all subject pairs:
1.4775544305207639e-38
Minimum normed rdmr across core ROIs and across all subject pairs:
0.8050828196815112
Maximum normed rdmr p-value across core ROIs and across all subject pairs:
0.0


In [67]:
mh_csv = pd.read_csv(MODELSUBJECT_CORRELATIONS_CSV_PATH)
mh_csv = mh_csv[
    (mh_csv['fmri_froi'].isin(core_frois)) &
    (mh_csv['fmri_tthreshold'] == tthreshold) &
    (mh_csv['model_name'].isin(to_model_names)) &
    (mh_csv['model_localizer'] == model_localizer) &
    (mh_csv['model_froi'].isin(core_frois)) &
    (mh_csv['model_tthreshold'] == tthreshold)
]

In [68]:
hh_lt_mh_stats_records = []
for metric in metrics:
    print(f'\n\nPaired sign-flip permutation test for whether model-subject {metric} is less than subject-subject {metric} (i.e., model is worse than ceiling):')
    for froi in core_frois:
        # print(f'\nFROI: {froi}')
        for model_name in to_model_names:
            hh_ceilings = []
            mh_corrs = []
            record = {'fmri_froi': froi, 'model_name': model_name, 'metric': metric}
            for subject in subjects:
                hh_ceiling = hh_corrs_csv[
                    (hh_corrs_csv['fmri_froi'] == froi) & 
                    (hh_corrs_csv['subject'] == subject)
                ][metric].values[0]
                hh_ceilings.append(hh_ceiling)
                mh_corr = mh_csv[
                    (mh_csv['model_name'] == model_name) &
                    (mh_csv['fmri_froi'] == froi) &
                    (mh_csv['model_froi'] == froi) &
                    (mh_csv['subject'] == subject)
                ][metric].values[0]
                mh_corrs.append(mh_corr)
            test_result = signflip_perm_test_against_ceiling(
                ceiling=hh_ceilings,
                model=mh_corrs,
                alternative='less',  # test model < ceiling
                statistic='median',
                exact=True, # N=8 -> exact 256 perms
                n_permutations=100_000,
                seed=0,
                add_one_correction=True,
            )
            record.update(test_result)
            # print(f'FROI: {froi}, model: {model_name}, metric: {metric}, p-value: {test_result["p_value"]}, median difference: {test_result["t_obs"]}, median subject-subject: {np.median(hh_ceilings)}, median model-subject: {np.median(mh_corrs)}')
            hh_lt_mh_stats_records.append(record)
hh_lt_mh_stats_df = pd.DataFrame(hh_lt_mh_stats_records)
print('Paired sign-flip permutation test results for whether model-subject correlations are less than subject-subject ceilings:')
display(hh_lt_mh_stats_df)
print('Number of models that are significantly worse than ceiling (p<0.05):')
print(f"{hh_lt_mh_stats_df[hh_lt_mh_stats_df['p_value'] < 0.05].shape[0]} out of {hh_lt_mh_stats_df.shape[0]} tests")



Paired sign-flip permutation test for whether model-subject normed_pearsonr is less than subject-subject normed_pearsonr (i.e., model is worse than ceiling):


Paired sign-flip permutation test for whether model-subject normed_rdmr is less than subject-subject normed_rdmr (i.e., model is worse than ceiling):
Paired sign-flip permutation test results for whether model-subject correlations are less than subject-subject ceilings:


,fmri_froi,model_name,metric,p_value,t_obs,diffs,n,n_perm,alternative,statistic
0,FFA,alexnetimagenet1kv1,normed_pearsonr,0.035019,-0.401091,"[-0.41795219068814093, -0.31287433844987167, -...",8,256,less,median
1,FFA,alexnetipclref01,normed_pearsonr,0.035019,-0.525682,"[-0.5332642600623466, -0.46420942594782005, -0...",8,256,less,median
2,FFA,alexnetbarlowtwins,normed_pearsonr,0.035019,-0.404394,"[-0.3738096341801882, -0.34089306390496243, -0...",8,256,less,median
3,FFA,vgg16imagenet1kv1,normed_pearsonr,0.035019,-0.483755,"[-0.5195933365285915, -0.4311107039115931, -0....",8,256,less,median
4,FFA,resnet18imagenet1kv1,normed_pearsonr,0.035019,-0.498459,"[-0.5080421000671715, -0.48312559051354814, -0...",8,256,less,median
...,...,...,...,...,...,...,...,...,...,...
205,EBA,vitb16synclr,normed_rdmr,0.035019,-0.955632,"[-0.9371567286322766, -0.9806781378067999, -0....",8,256,less,median
206,EBA,vitb14dinov2,normed_rdmr,0.035019,-0.938945,"[-0.9283647051118438, -0.9466325768561279, -0....",8,256,less,median
207,EBA,vitb14dinov2reg,normed_rdmr,0.035019,-0.974298,"[-0.9803653942847139, -0.9682310839148522, -0....",8,256,less,median
208,EBA,vitl14openaiclip,normed_rdmr,0.035019,-0.938016,"[-0.9414986457608717, -0.9596448236570652, -0....",8,256,less,median


Number of models that are significantly worse than ceiling (p<0.05):
210 out of 210 tests


In [69]:
hh_lt_mh_stats_df_pop = hh_lt_mh_stats_df.copy()
# Add a new column for median_diffs
hh_lt_mh_stats_df_pop['median_diff'] = hh_lt_mh_stats_df_pop['diffs'].apply(np.median)
print('Running a population-level test of whether model correlations are less than ceiling correlations, across all models and ROIs')
print('Wilcoxon signed-rank test on median differences across all models and ROIs, againts the null hypothesis that the median difference is zero (i.e., model = ceiling):')
for froi in core_frois:
    print('\nFROI:', froi)
    for metric in metrics:
        
        median_diffs = hh_lt_mh_stats_df_pop[
            (hh_lt_mh_stats_df_pop['fmri_froi'] == froi) &
            (hh_lt_mh_stats_df_pop['metric'] == metric)
        ]['median_diff'].values
        stat, p_value = stats.wilcoxon(median_diffs, alternative='less', zero_method='wilcox')
        print(f'FROI: {froi}, metric: {metric}, Wilcoxon stat: {stat}, p-value: {p_value}, median of median differences: {np.median(median_diffs)}')

Running a population-level test of whether model correlations are less than ceiling correlations, across all models and ROIs
Wilcoxon signed-rank test on median differences across all models and ROIs, againts the null hypothesis that the median difference is zero (i.e., model = ceiling):

FROI: FFA
FROI: FFA, metric: normed_pearsonr, Wilcoxon stat: 0.0, p-value: 2.9103830456733704e-11, median of median differences: -0.4325429545226097
FROI: FFA, metric: normed_rdmr, Wilcoxon stat: 0.0, p-value: 2.9103830456733704e-11, median of median differences: -0.6374903973192887

FROI: PPA
FROI: PPA, metric: normed_pearsonr, Wilcoxon stat: 0.0, p-value: 2.9103830456733704e-11, median of median differences: -0.858943811685259
FROI: PPA, metric: normed_rdmr, Wilcoxon stat: 0.0, p-value: 2.9103830456733704e-11, median of median differences: -0.8835981838910055

FROI: EBA
FROI: EBA, metric: normed_pearsonr, Wilcoxon stat: 0.0, p-value: 2.9103830456733704e-11, median of median differences: -0.489258507

### Voxel-tuning matched

In [70]:
tthreshold = 7
model_localizer = 'nsd'

SUBJECTSUBJECT_CORRELATIONS_CSV_PATH = f'{RESULTS_DIR}/subjectsubject_correlations_nsd515.csv'
MODELSUBJECT_CORRELATIONS_CSV_PATH = f'{RESULTS_DIR}/modelsubject_correlations_nsd515.csv'

metrics = ['normed_pearsonr', 'normed_rdmr']
subjects = list(range(1, 9)) 

In [71]:
hh_csv = pd.read_csv(SUBJECTSUBJECT_CORRELATIONS_CSV_PATH)
hh_csv_subset = hh_csv[
    (hh_csv['fmri_froi'].isin(core_frois)) &
    (hh_csv['fmri_tthreshold'] == tthreshold)
]

print(f'Minimum normed pearsonr across core ROIs and across all subject pairs:')
print(hh_csv_subset['normed_pearsonr'].min().item())
print(f'Maximum normed pearsonr p-value across core ROIs and across all subject pairs:')
print(hh_csv_subset['pvalue_pearsonr'].max().item())
print(f'Minimum normed rdmr across core ROIs and across all subject pairs:')
print(hh_csv_subset['normed_rdmr'].min().item())
print(f'Maximum normed rdmr p-value across core ROIs and across all subject pairs:')
print(hh_csv_subset['pvalue_rdmr'].max().item())


hh_corrs_csv_records = []
median_correlations_records = []
for froi in core_frois:
    record = {'fmri_froi': froi}
    hh_median_correlations = {metric: [] for metric in metrics}
    for subject in subjects:
        hh_record = {'fmri_froi': froi, 'subject': subject}
        hh_csv_ = hh_csv[
            (hh_csv['fmri_froi'] == froi) &
            (hh_csv['fmri_tthreshold'] == tthreshold) &
            (
                (hh_csv['subject1'] == subject) | 
                (hh_csv['subject2'] == subject)
            )
        ]
        assert len(hh_csv_) == 7, f'Expected 7 rows for subject {subject} and froi {froi}, but got {len(hh_csv_)}'
        for metric in metrics:
            median_metric = hh_csv_[metric].median()
            hh_record[metric] = median_metric
            hh_median_correlations[metric].append(median_metric)
        hh_corrs_csv_records.append(hh_record)
    for metric in metrics:
        record[f'{metric}_median'] = np.median(hh_median_correlations[metric])
    median_correlations_records.append(record)
hh_corrs_csv = pd.DataFrame(hh_corrs_csv_records)
hh_median_csv = pd.DataFrame(median_correlations_records)

Minimum normed pearsonr across core ROIs and across all subject pairs:
0.8727959208707898
Maximum normed pearsonr p-value across core ROIs and across all subject pairs:
1.4775544305207639e-38
Minimum normed rdmr across core ROIs and across all subject pairs:
0.8050828196815112
Maximum normed rdmr p-value across core ROIs and across all subject pairs:
0.0


In [72]:
mh_csv = pd.read_csv(MODELSUBJECT_CORRELATIONS_CSV_PATH)
mh_csv = mh_csv[
    (mh_csv['fmri_froi'].isin(core_frois)) &
    (mh_csv['fmri_tthreshold'] == tthreshold) &
    (mh_csv['model_name'].isin(to_model_names)) &
    (mh_csv['model_localizer'] == model_localizer) &
    (mh_csv['model_froi'].isin(core_frois)) &
    (mh_csv['model_tthreshold'] == tthreshold)
]

In [74]:
hh_lt_mh_stats_records = []
for metric in metrics:
    print(f'\n\nPaired sign-flip permutation test for whether model-subject {metric} is less than subject-subject {metric} (i.e., model is worse than ceiling):')
    for froi in core_frois:
        # print(f'\nFROI: {froi}')
        for model_name in to_model_names:
            hh_ceilings = []
            mh_corrs = []
            record = {'fmri_froi': froi, 'model_name': model_name, 'metric': metric}
            for subject in subjects:
                hh_ceiling = hh_corrs_csv[
                    (hh_corrs_csv['fmri_froi'] == froi) & 
                    (hh_corrs_csv['subject'] == subject)
                ][metric].values[0]
                hh_ceilings.append(hh_ceiling)
                mh_corr = mh_csv[
                    (mh_csv['model_name'] == model_name) &
                    (mh_csv['fmri_froi'] == froi) &
                    (mh_csv['model_froi'] == froi) &
                    (mh_csv['subject'] == subject)
                ][metric].values[0]
                mh_corrs.append(mh_corr)
            test_result = signflip_perm_test_against_ceiling(
                ceiling=hh_ceilings,
                model=mh_corrs,
                alternative='less',  # test model < ceiling
                statistic='median',
                exact=True, # N=8 -> exact 256 perms
                n_permutations=100_000,
                seed=0,
                add_one_correction=True,
            )
            record.update(test_result)
            # print(f'FROI: {froi}, model: {model_name}, metric: {metric}, p-value: {test_result["p_value"]}, median difference: {test_result["t_obs"]}, median subject-subject: {np.median(hh_ceilings)}, median model-subject: {np.median(mh_corrs)}')
            hh_lt_mh_stats_records.append(record)
hh_lt_mh_stats_df = pd.DataFrame(hh_lt_mh_stats_records)
print('Paired sign-flip permutation test results for whether model-subject correlations are less than subject-subject ceilings:')
display(hh_lt_mh_stats_df)
print('Number of models that are significantly worse than ceiling (p<0.05):')
print(f"{hh_lt_mh_stats_df[hh_lt_mh_stats_df['p_value'] < 0.05].shape[0]} out of {hh_lt_mh_stats_df.shape[0]} tests")



Paired sign-flip permutation test for whether model-subject normed_pearsonr is less than subject-subject normed_pearsonr (i.e., model is worse than ceiling):


Paired sign-flip permutation test for whether model-subject normed_rdmr is less than subject-subject normed_rdmr (i.e., model is worse than ceiling):
Paired sign-flip permutation test results for whether model-subject correlations are less than subject-subject ceilings:


,fmri_froi,model_name,metric,p_value,t_obs,diffs,n,n_perm,alternative,statistic
0,FFA,alexnetimagenet1kv1,normed_pearsonr,0.035019,-0.271080,"[-0.2629818612602207, -0.23972247949243108, -0...",8,256,less,median
1,FFA,alexnetipclref01,normed_pearsonr,0.035019,-0.345783,"[-0.3239963462193529, -0.2888533883545651, -0....",8,256,less,median
2,FFA,alexnetbarlowtwins,normed_pearsonr,0.035019,-0.298976,"[-0.23466183885208047, -0.2245721693442475, -0...",8,256,less,median
3,FFA,vgg16imagenet1kv1,normed_pearsonr,0.035019,-0.214040,"[-0.16256700541933988, -0.1902965296804493, -0...",8,256,less,median
4,FFA,resnet18imagenet1kv1,normed_pearsonr,0.035019,-0.323351,"[-0.23854018132032584, -0.3316549179602962, -0...",8,256,less,median
...,...,...,...,...,...,...,...,...,...,...
205,EBA,vitb16synclr,normed_rdmr,0.035019,-0.839208,"[-0.8357896734062862, -0.842626032801794, -0.8...",8,256,less,median
206,EBA,vitb14dinov2,normed_rdmr,0.035019,-0.704913,"[-0.6837700298402687, -0.7073510133387348, -0....",8,256,less,median
207,EBA,vitb14dinov2reg,normed_rdmr,0.035019,-0.892598,"[-0.8891958611073291, -0.8959998723491163, -0....",8,256,less,median
208,EBA,vitl14openaiclip,normed_rdmr,0.035019,-0.722578,"[-0.7225826895564631, -0.750745941328216, -0.7...",8,256,less,median


Number of models that are significantly worse than ceiling (p<0.05):
210 out of 210 tests


In [77]:
hh_lt_mh_stats_df_pop = hh_lt_mh_stats_df.copy()
# Add a new column for median_diffs
hh_lt_mh_stats_df_pop['median_diff'] = hh_lt_mh_stats_df_pop['diffs'].apply(np.median)
print('Running a population-level test of whether model correlations are less than ceiling correlations, across all models and ROIs')
print('Wilcoxon signed-rank test on median differences across all models and ROIs, againts the null hypothesis that the median difference is zero (i.e., model = ceiling):')
for froi in core_frois:
    print('\nFROI:', froi)
    for metric in metrics:
        
        median_diffs = hh_lt_mh_stats_df_pop[
            (hh_lt_mh_stats_df_pop['fmri_froi'] == froi) &
            (hh_lt_mh_stats_df_pop['metric'] == metric)
        ]['median_diff'].values
        stat, p_value = stats.wilcoxon(median_diffs, alternative='less', zero_method='wilcox')
        print(f'FROI: {froi}, metric: {metric}, Wilcoxon stat: {stat}, p-value: {p_value}, median of median differences: {np.median(median_diffs)}')

Running a population-level test of whether model correlations are less than ceiling correlations, across all models and ROIs
Wilcoxon signed-rank test on median differences across all models and ROIs, againts the null hypothesis that the median difference is zero (i.e., model = ceiling):

FROI: FFA
FROI: FFA, metric: normed_pearsonr, Wilcoxon stat: 0.0, p-value: 2.9103830456733704e-11, median of median differences: -0.22427597911854324
FROI: FFA, metric: normed_rdmr, Wilcoxon stat: 0.0, p-value: 2.9103830456733704e-11, median of median differences: -0.5005595450325291

FROI: EBA
FROI: EBA, metric: normed_pearsonr, Wilcoxon stat: 0.0, p-value: 2.9103830456733704e-11, median of median differences: -0.25961917744613666
FROI: EBA, metric: normed_rdmr, Wilcoxon stat: 0.0, p-value: 2.9103830456733704e-11, median of median differences: -0.6832624162734193

FROI: PPA
FROI: PPA, metric: normed_pearsonr, Wilcoxon stat: 0.0, p-value: 2.9103830456733704e-11, median of median differences: -0.223737

## fROIs

In [ ]:
tthreshold = 7
model_localizer = 'grillspector'

SUBJECTSUBJECT_CORRELATIONS_CSV_PATH = f'{RESULTS_DIR}/subjectsubject_correlations_nsd515.csv'
MODELSUBJECT_CORRELATIONS_CSV_PATH = f'{RESULTS_DIR}/modelsubject_correlations_nsd515.csv'

metrics = ['normed_pearsonr', 'normed_rdmr']
subjects = list(range(1, 9)) 

In [ ]:
hh_csv = pd.read_csv(SUBJECTSUBJECT_CORRELATIONS_CSV_PATH)
hh_csv_subset = hh_csv[
    (hh_csv['fmri_froi'].isin(all_extra_frois)) &
    (hh_csv['fmri_tthreshold'] == tthreshold)
]

print(f'Minimum normed pearsonr across core ROIs and across all subject pairs:')
print(hh_csv_subset['normed_pearsonr'].min().item())
print(f'Maximum normed pearsonr p-value across core ROIs and across all subject pairs:')
print(hh_csv_subset['pvalue_pearsonr'].max().item())
print(f'Minimum normed rdmr across core ROIs and across all subject pairs:')
print(hh_csv_subset['normed_rdmr'].min().item())
print(f'Maximum normed rdmr p-value across core ROIs and across all subject pairs:')
print(hh_csv_subset['pvalue_rdmr'].max().item())


hh_corrs_csv_records = []
median_correlations_records = []
for froi in all_extra_frois:
    record = {'fmri_froi': froi}
    hh_median_correlations = {metric: [] for metric in metrics}
    for subject in subjects:
        hh_record = {'fmri_froi': froi, 'subject': subject}
        hh_csv_ = hh_csv[
            (hh_csv['fmri_froi'] == froi) &
            (hh_csv['fmri_tthreshold'] == tthreshold) &
            (
                (hh_csv['subject1'] == subject) | 
                (hh_csv['subject2'] == subject)
            )
        ]
        assert len(hh_csv_) == 7, f'Expected 7 rows for subject {subject} and froi {froi}, but got {len(hh_csv_)}'
        for metric in metrics:
            median_metric = hh_csv_[metric].median()
            hh_record[metric] = median_metric
            hh_median_correlations[metric].append(median_metric)
        hh_corrs_csv_records.append(hh_record)
    for metric in metrics:
        record[f'{metric}_median'] = np.median(hh_median_correlations[metric])
    median_correlations_records.append(record)
hh_corrs_csv = pd.DataFrame(hh_corrs_csv_records)
hh_median_csv = pd.DataFrame(median_correlations_records)

Minimum normed pearsonr across core ROIs and across all subject pairs:
0.4454693750465781
Maximum normed pearsonr p-value across core ROIs and across all subject pairs:
6.039663461988632e-07
Minimum normed rdmr across core ROIs and across all subject pairs:
0.3308773116532983
Maximum normed rdmr p-value across core ROIs and across all subject pairs:
1.7269814814928552e-51


In [59]:
mh_csv = pd.read_csv(MODELSUBJECT_CORRELATIONS_CSV_PATH)
mh_csv = mh_csv[
    (mh_csv['fmri_froi'].isin(all_extra_frois)) &
    (mh_csv['fmri_tthreshold'] == tthreshold) &
    (mh_csv['model_name'].isin(to_model_names)) &
    (mh_csv['model_localizer'] == model_localizer) &
    (mh_csv['model_froi'].isin(core_frois + ['VWFA'])) &
    (mh_csv['model_tthreshold'] == tthreshold)
]

In [62]:
froi_pairing_dict = { # fmri_froi: model_froi
    'OFA': 'FFA',
    'FBA': 'EBA',
    'OPA': 'PPA',
    'VWFA': 'VWFA',
    'OVWFA': 'VWFA',
}

hh_lt_mh_stats_records = []
for metric in metrics:
    print(f'\n\nPaired sign-flip permutation test for whether model-subject {metric} is less than subject-subject {metric} (i.e., model is worse than ceiling):')
    for froi in all_extra_frois:
        # print(f'\nFROI: {froi}')
        for model_name in to_model_names:
            hh_ceilings = []
            mh_corrs = []
            record = {'fmri_froi': froi, 'model_name': model_name, 'metric': metric}
            for subject in subjects:
                hh_ceiling = hh_corrs_csv[
                    (hh_corrs_csv['fmri_froi'] == froi) & 
                    (hh_corrs_csv['subject'] == subject)
                ][metric].values[0]
                hh_ceilings.append(hh_ceiling)
                mh_corr = mh_csv[
                    (mh_csv['model_name'] == model_name) &
                    (mh_csv['fmri_froi'] == froi) &
                    (mh_csv['model_froi'] == froi_pairing_dict[froi]) &
                    (mh_csv['subject'] == subject)
                ][metric].values[0]
                mh_corrs.append(mh_corr)
            test_result = signflip_perm_test_against_ceiling(
                ceiling=hh_ceilings,
                model=mh_corrs,
                alternative='less',  # test model < ceiling
                statistic='median',
                exact=True, # N=8 -> exact 256 perms
                n_permutations=100_000,
                seed=0,
                add_one_correction=True,
            )
            record.update(test_result)
            # print(f'FROI: {froi}, model: {model_name}, metric: {metric}, p-value: {test_result["p_value"]}, median difference: {test_result["t_obs"]}, median subject-subject: {np.median(hh_ceilings)}, median model-subject: {np.median(mh_corrs)}')
            hh_lt_mh_stats_records.append(record)
hh_lt_mh_stats_df = pd.DataFrame(hh_lt_mh_stats_records)
print('Paired sign-flip permutation test results for whether model-subject correlations are less than subject-subject ceilings:')
display(hh_lt_mh_stats_df)
print('Number of models that are significantly worse than ceiling (p<0.05):')
print(f"{hh_lt_mh_stats_df[hh_lt_mh_stats_df['p_value'] < 0.05].shape[0]} out of {hh_lt_mh_stats_df.shape[0]} tests")



Paired sign-flip permutation test for whether model-subject normed_pearsonr is less than subject-subject normed_pearsonr (i.e., model is worse than ceiling):


Paired sign-flip permutation test for whether model-subject normed_rdmr is less than subject-subject normed_rdmr (i.e., model is worse than ceiling):
Paired sign-flip permutation test results for whether model-subject correlations are less than subject-subject ceilings:


,fmri_froi,model_name,metric,p_value,t_obs,diffs,n,n_perm,alternative,statistic
0,OFA,alexnetimagenet1kv1,normed_pearsonr,0.035019,-0.329298,"[-0.2116968506807958, -0.08866199977233025, -0...",8,256,less,median
1,OFA,alexnetipclref01,normed_pearsonr,0.035019,-0.435810,"[-0.23181119219956614, -0.18137284538379145, -...",8,256,less,median
2,OFA,alexnetbarlowtwins,normed_pearsonr,0.035019,-0.365162,"[-0.30448881521146687, -0.22175959758764424, -...",8,256,less,median
3,OFA,vgg16imagenet1kv1,normed_pearsonr,0.035019,-0.260740,"[-0.10472367552377315, -0.0190105913000308, -0...",8,256,less,median
4,OFA,resnet18imagenet1kv1,normed_pearsonr,0.035019,-0.406860,"[-0.19270580425150297, -0.07956619238347051, -...",8,256,less,median
...,...,...,...,...,...,...,...,...,...,...
345,OVWFA,vitb16synclr,normed_rdmr,0.035019,-0.618831,"[-0.42977246021242366, -0.5545757751024996, -0...",8,256,less,median
346,OVWFA,vitb14dinov2,normed_rdmr,0.035019,-0.705914,"[-0.4907024482015936, -0.6592553393000522, -0....",8,256,less,median
347,OVWFA,vitb14dinov2reg,normed_rdmr,0.035019,-0.523751,"[-0.3707347664203835, -0.4634345427760074, -0....",8,256,less,median
348,OVWFA,vitl14openaiclip,normed_rdmr,0.035019,-0.608951,"[-0.6057149277986262, -0.6121880256590164, -0....",8,256,less,median


Number of models that are significantly worse than ceiling (p<0.05):
350 out of 350 tests


In [63]:
hh_lt_mh_stats_df_pop = hh_lt_mh_stats_df.copy()
# Add a new column for median_diffs
hh_lt_mh_stats_df_pop['median_diff'] = hh_lt_mh_stats_df_pop['diffs'].apply(np.median)
print('Running a population-level test of whether model correlations are less than ceiling correlations, across all models and ROIs')
print('Wilcoxon signed-rank test on median differences across all models and ROIs, againts the null hypothesis that the median difference is zero (i.e., model = ceiling):')
for froi in all_extra_frois:
    print('\nFROI:', froi)
    for metric in metrics:
        
        median_diffs = hh_lt_mh_stats_df_pop[
            (hh_lt_mh_stats_df_pop['fmri_froi'] == froi) &
            (hh_lt_mh_stats_df_pop['metric'] == metric)
        ]['median_diff'].values
        stat, p_value = stats.wilcoxon(median_diffs, alternative='less', zero_method='wilcox')
        print(f'FROI: {froi}, metric: {metric}, Wilcoxon stat: {stat}, p-value: {p_value}, median of median differences: {np.median(median_diffs)}')

Running a population-level test of whether model correlations are less than ceiling correlations, across all models and ROIs
Wilcoxon signed-rank test on median differences across all models and ROIs, againts the null hypothesis that the median difference is zero (i.e., model = ceiling):

FROI: OFA
FROI: OFA, metric: normed_pearsonr, Wilcoxon stat: 0.0, p-value: 2.9103830456733704e-11, median of median differences: -0.34248481707334905
FROI: OFA, metric: normed_rdmr, Wilcoxon stat: 0.0, p-value: 2.9103830456733704e-11, median of median differences: -0.5573199802756765

FROI: FBA
FROI: FBA, metric: normed_pearsonr, Wilcoxon stat: 0.0, p-value: 2.9103830456733704e-11, median of median differences: -0.31071510168934674
FROI: FBA, metric: normed_rdmr, Wilcoxon stat: 0.0, p-value: 2.9103830456733704e-11, median of median differences: -0.7226448214670322

FROI: OPA
FROI: OPA, metric: normed_pearsonr, Wilcoxon stat: 0.0, p-value: 2.9103830456733704e-11, median of median differences: -0.332598

# Section 2.5

In [80]:
em_csv = pd.read_csv(f'{RESULTS_DIR}/layer_froi_lesion_ems_fixedalpha.csv')
em_csv2_tthresholds = pd.read_csv(f'{RESULTS_DIR}/nsd515_lesion_ems_fixedalpha.csv')

### Mean performance drop

In [101]:
mapper = 'ridge'
alpha = 0.1

em_csv_subset = em_csv[
    (em_csv['mapper'] == mapper) &
    (em_csv['alpha'] == alpha) &
    (em_csv['model_froi'].isin(core_frois)) &
    (em_csv['model_localizer'] == model_localizer) &
    (em_csv['model_tthreshold'] == tthreshold)
].copy()

# Estimate mean and median of difference between layer_raw_pearsonr and lesion_raw_pearsonr, as well as for rdmr
em_csv_subset['delta_pearsonr'] = (em_csv_subset['layer_raw_pearsonr'] - em_csv_subset['lesion_raw_pearsonr'])/em_csv_subset['fmri_splithalf_pearsonr']*100
em_csv_subset['delta_rdmr'] = (em_csv_subset['layer_raw_rdmr'] - em_csv_subset['lesion_raw_rdmr'])/em_csv_subset['fmri_splithalf_rdmr']*100
em_csv_subset_grouped = (
    em_csv_subset
    .groupby(['model', 'fmri_froi'])
    .agg(
        delta_pearsonr_lesion_median=('delta_pearsonr', 'median'),
        delta_pearsonr_lesion_mean=('delta_pearsonr', 'mean'),
        delta_rdmr_lesion_median=('delta_rdmr', 'median'),
        delta_rdmr_lesion_mean=('delta_rdmr', 'mean'),
    )
    .reset_index()
)
print('Delta pearsonr and rdmr between layer and lesion (grouped by ROI):')
display(em_csv_subset_grouped)
print(f'Median delta pearsonr (layer - lesion): {em_csv_subset_grouped["delta_pearsonr_lesion_median"].median()}')
print(f'Mean delta pearsonr (layer - lesion): {em_csv_subset_grouped["delta_pearsonr_lesion_mean"].mean()}')
print(f'Median delta rdmr (layer - lesion): {em_csv_subset_grouped["delta_rdmr_lesion_median"].median()}')
print(f'Mean delta rdmr (layer - lesion): {em_csv_subset_grouped["delta_rdmr_lesion_mean"].mean()}')
print('\n')

# Estimate the same between layer_raw_pearsonr and froi_raw_pearsonr, as well as for rdmr
em_csv_subset['delta_pearsonr_froi'] = (em_csv_subset['layer_raw_pearsonr'] - em_csv_subset['froi_raw_pearsonr'])/em_csv_subset['fmri_splithalf_pearsonr']*100
em_csv_subset['delta_rdmr_froi'] = (em_csv_subset['layer_raw_rdmr'] - em_csv_subset['froi_raw_rdmr'])/em_csv_subset['fmri_splithalf_rdmr']*100
em_csv_subset_grouped = (
    em_csv_subset
    .groupby(['model', 'fmri_froi'])
    .agg(
        delta_pearsonr_froi_median=('delta_pearsonr_froi', 'median'),
        delta_pearsonr_froi_mean=('delta_pearsonr_froi', 'mean'),
        delta_rdmr_froi_median=('delta_rdmr_froi', 'median'),
        delta_rdmr_froi_mean=('delta_rdmr_froi', 'mean'),
    )
    .reset_index()
)
print(f'Median delta pearsonr (layer vs froi): {em_csv_subset_grouped["delta_pearsonr_froi_median"].median()}')
print(f'Mean delta pearsonr (layer vs froi): {em_csv_subset_grouped["delta_pearsonr_froi_mean"].mean()}')
print(f'Median delta rdmr (layer vs froi): {em_csv_subset_grouped["delta_rdmr_froi_median"].median()}')
print(f'Mean delta rdmr (layer vs froi): {em_csv_subset_grouped["delta_rdmr_froi_mean"].mean()}')

Delta pearsonr and rdmr between layer and lesion (grouped by ROI):


,model,fmri_froi,delta_pearsonr_lesion_median,delta_pearsonr_lesion_mean,delta_rdmr_lesion_median,delta_rdmr_lesion_mean
0,alexnetbarlowtwins,EBA,0.172063,0.150470,0.132157,0.112881
1,alexnetbarlowtwins,FFA,1.144460,1.259357,2.040749,2.040385
2,alexnetbarlowtwins,PPA,0.243741,0.239526,0.764168,0.860932
3,alexnetimagenet1kv1,EBA,-0.076951,-0.070807,0.125342,0.128114
4,alexnetimagenet1kv1,FFA,2.465315,2.718056,6.613113,7.079073
...,...,...,...,...,...,...
100,vitl14kosmos2,FFA,0.651294,0.608688,1.474969,1.701241
101,vitl14kosmos2,PPA,0.092853,0.084200,0.244774,0.231852
102,vitl14openaiclip,EBA,0.054037,0.052345,0.127599,0.126377
103,vitl14openaiclip,FFA,1.206838,1.176538,3.420385,3.348226


Median delta pearsonr (layer - lesion): 0.22230869633206377
Mean delta pearsonr (layer - lesion): 0.43291650939645926
Median delta rdmr (layer - lesion): 0.47458896839125564
Mean delta rdmr (layer - lesion): 1.194150102388364


Median delta pearsonr (layer vs froi): 18.372445979877554
Mean delta pearsonr (layer vs froi): 22.93605890358838
Median delta rdmr (layer vs froi): 30.582567997250926
Mean delta rdmr (layer vs froi): 34.02118678724134


### Stats

In [93]:
test_normality = False

mapper = 'ridge'
alpha = 0.1

em_csv_subset = em_csv[
    (em_csv['mapper'] == mapper) &
    (em_csv['alpha'] == alpha) &
    (em_csv['model_froi'].isin(core_frois)) &
    (em_csv['model_localizer'] == model_localizer) &
    (em_csv['model_tthreshold'] == tthreshold)
]

em_metrics = ['raw_pearsonr', 'raw_rdmr']

for metric in em_metrics:
    print(f'Metric: {metric}')
    for froi in core_frois:
        froi_subset = em_csv_subset[
            (em_csv_subset['model_froi'] == froi) &
            (em_csv_subset['fmri_froi'] == froi)
        ].copy()
        # Add two new columns for the metric of interest
        # First is the normed layer - lesion metric (with-in subject delta)
        # Second is the normed layer - froi metric (with-in subject delta)
        froi_subset.loc[:, f'layer-lesion_{metric}'] = froi_subset[f'layer_{metric}'] - froi_subset[f'lesion_{metric}']
        froi_subset.loc[:, f'layer-froi_{metric}'] = froi_subset[f'layer_{metric}'] - froi_subset[f'froi_{metric}']
        # Group by model and take median across subjects for each metric
        froi_subset_grouped = (
            froi_subset
            .groupby(['model'])
            .agg(
                **{
                    # f'layer_{metric}': (f'layer_{metric}', 'median'),
                    # f'lesion_{metric}': (f'lesion_{metric}', 'median'),
                    # f'froi_{metric}': (f'froi_{metric}', 'median'),
                    f'layer-lesion_{metric}': (f'layer-lesion_{metric}', 'median'),
                    f'layer-froi_{metric}': (f'layer-froi_{metric}', 'median'),
                }
            )
            .reset_index()
        )
        assert len(froi_subset_grouped) == len(to_model_names), f'Expected {len(to_model_names)} models in the grouped subset for froi {froi}, but got {len(froi_subset_grouped)}'
        # layer_metric = froi_subset_grouped[f'layer_{metric}']
        # lesion_metric = froi_subset_grouped[f'lesion_{metric}']
        # froi_metric = froi_subset_grouped[f'froi_{metric}']
        layer_lesion_metric = froi_subset_grouped[f'layer-lesion_{metric}']
        layer_froi_metric = froi_subset_grouped[f'layer-froi_{metric}']
        print(f'  FROI: {froi}')
        if test_normality:
            # Do a test of normality for each metric
            # layer_normality_pval = stats.shapiro(layer_metric).pvalue
            # lesion_normality_pval = stats.shapiro(lesion_metric).pvalue
            # froi_normality_pval = stats.shapiro(froi_metric).pvalue
            layer_lesion_normality_pval = stats.shapiro(layer_lesion_metric).pvalue
            layer_froi_normality_pval = stats.shapiro(layer_froi_metric).pvalue
            # Print p-values and the interpretation
            # print(f'    Layer {metric} normality p-value: {layer_normality_pval} -> {"normal" if layer_normality_pval >= 0.05 else "not normal"}')
            # print(f'    Lesion {metric} normality p-value: {lesion_normality_pval} -> {"normal" if lesion_normality_pval >= 0.05 else "not normal"}')
            # print(f'    FROI {metric} normality p-value: {froi_normality_pval} -> {"normal" if froi_normality_pval >= 0.05 else "not normal"}')
            print(f'    Layer-Lesion {metric} normality p-value: {layer_lesion_normality_pval} -> {"normal" if layer_lesion_normality_pval >= 0.05 else "not normal"}')
            print(f'    Layer-FROI {metric} normality p-value: {layer_froi_normality_pval} -> {"normal" if layer_froi_normality_pval >= 0.05 else "not normal"}')
        # Do a paired Wilcoxon signed-rank test between layer and lesion metrics, and between layer and froi metrics
        # wilcoxon_layer_lesion_stat, wilcoxon_layer_lesion_pval = stats.wilcoxon(layer_metric, lesion_metric, alternative='greater')
        # wilcoxon_layer_froi_stat, wilcoxon_layer_froi_pval = stats.wilcoxon(layer_metric, froi_metric, alternative='greater')
        # print(f'    Wilcoxon layer vs lesion {metric}: stat={wilcoxon_layer_lesion_stat}, p-value={wilcoxon_layer_lesion_pval}')
        # print(f'    Wilcoxon layer vs froi {metric}: stat={wilcoxon_layer_froi_stat}, p-value={wilcoxon_layer_froi_pval}')
        # Do a paired wilcoxon between layer and lesion metrics, and between layer and froi metrics
        wilcoxon_layer_lesion_stat, wilcoxon_layer_lesion_pval = stats.wilcoxon(layer_lesion_metric, alternative='greater')
        wilcoxon_layer_froi_stat, wilcoxon_layer_froi_pval = stats.wilcoxon(layer_froi_metric, alternative='greater')
        print(f'    Wilcoxon layer-lesion {metric}: stat={wilcoxon_layer_lesion_stat}, p-value={wilcoxon_layer_lesion_pval}')
        print(f'    Wilcoxon layer-froi {metric}: stat={wilcoxon_layer_froi_stat}, p-value={wilcoxon_layer_froi_pval}')
    print('\n')

Metric: raw_pearsonr
  FROI: FFA
    Wilcoxon layer-lesion raw_pearsonr: stat=630.0, p-value=2.9103830456733704e-11
    Wilcoxon layer-froi raw_pearsonr: stat=625.0, p-value=2.9103830456733704e-10
  FROI: PPA
    Wilcoxon layer-lesion raw_pearsonr: stat=613.0, p-value=6.024492904543877e-09
    Wilcoxon layer-froi raw_pearsonr: stat=619.0, p-value=1.6007106751203537e-09
  FROI: EBA
    Wilcoxon layer-lesion raw_pearsonr: stat=541.0, p-value=4.591900506056845e-05
    Wilcoxon layer-froi raw_pearsonr: stat=610.0, p-value=1.0797521099448204e-08


Metric: raw_rdmr
  FROI: FFA
    Wilcoxon layer-lesion raw_rdmr: stat=630.0, p-value=2.9103830456733704e-11
    Wilcoxon layer-froi raw_rdmr: stat=616.0, p-value=3.2014213502407074e-09
  FROI: PPA
    Wilcoxon layer-lesion raw_rdmr: stat=616.0, p-value=3.2014213502407074e-09
    Wilcoxon layer-froi raw_rdmr: stat=620.0, p-value=1.2514647096395493e-09
  FROI: EBA
    Wilcoxon layer-lesion raw_rdmr: stat=607.0, p-value=1.862645149230957e-08
    Wilc

#### Legacy (incorrect: median across subjects then delta)

In [ ]:
test_normality = False

mapper = 'ridge'
alpha = 0.1

em_csv_subset = em_csv[
    (em_csv['mapper'] == mapper) &
    (em_csv['alpha'] == alpha) &
    (em_csv['model_froi'].isin(core_frois)) &
    (em_csv['model_localizer'] == model_localizer) &
    (em_csv['model_tthreshold'] == tthreshold)
]

em_metrics = ['raw_pearsonr', 'raw_rdmr']

for metric in em_metrics:
    print(f'Metric: {metric}')
    for froi in core_frois:
        froi_subset = em_csv_subset[
            (em_csv_subset['model_froi'] == froi) &
            (em_csv_subset['fmri_froi'] == froi)
        ]
        # Group by model and take median across subjects for each metric
        froi_subset_grouped = (
            froi_subset
            .groupby(['model'])
            .agg(
                **{
                    f'layer_{metric}': (f'layer_{metric}', 'median'),
                    f'lesion_{metric}': (f'lesion_{metric}', 'median'),
                    f'froi_{metric}': (f'froi_{metric}', 'median'),
                }
            )
            .reset_index()
        )
        assert len(froi_subset_grouped) == len(to_model_names), f'Expected {len(to_model_names)} models in the grouped subset for froi {froi}, but got {len(froi_subset_grouped)}'
        layer_metric = froi_subset_grouped[f'layer_{metric}']
        lesion_metric = froi_subset_grouped[f'lesion_{metric}']
        froi_metric = froi_subset_grouped[f'froi_{metric}']
        print(f'  FROI: {froi}')
        if test_normality:
            # Do a test of normality for each metric
            layer_normality_pval = stats.shapiro(layer_metric).pvalue
            lesion_normality_pval = stats.shapiro(lesion_metric).pvalue
            froi_normality_pval = stats.shapiro(froi_metric).pvalue
            # Print p-values and the interpretation
            print(f'    Layer {metric} normality p-value: {layer_normality_pval} -> {"normal" if layer_normality_pval >= 0.05 else "not normal"}')
            print(f'    Lesion {metric} normality p-value: {lesion_normality_pval} -> {"normal" if lesion_normality_pval >= 0.05 else "not normal"}')
            print(f'    FROI {metric} normality p-value: {froi_normality_pval} -> {"normal" if froi_normality_pval >= 0.05 else "not normal"}')
        # Do a paired Wilcoxon signed-rank test between layer and lesion metrics, and between layer and froi metrics
        wilcoxon_layer_lesion_stat, wilcoxon_layer_lesion_pval = stats.wilcoxon(layer_metric, lesion_metric, alternative='greater')
        wilcoxon_layer_froi_stat, wilcoxon_layer_froi_pval = stats.wilcoxon(layer_metric, froi_metric, alternative='greater')
        print(f'    Wilcoxon layer vs lesion {metric}: stat={wilcoxon_layer_lesion_stat}, p-value={wilcoxon_layer_lesion_pval}')
        print(f'    Wilcoxon layer vs froi {metric}: stat={wilcoxon_layer_froi_stat}, p-value={wilcoxon_layer_froi_pval}')
    print('\n')

Metric: raw_pearsonr
  FROI: FFA
    Wilcoxon layer vs lesion raw_pearsonr: stat=9854.0, p-value=7.135600655378373e-25
    Wilcoxon layer vs froi raw_pearsonr: stat=9744.0, p-value=7.386606824496166e-24
  FROI: PPA
    Wilcoxon layer vs lesion raw_pearsonr: stat=9299.0, p-value=5.557665743376074e-20
    Wilcoxon layer vs froi raw_pearsonr: stat=9606.0, p-value=1.2881810440656058e-22
  FROI: EBA
    Wilcoxon layer vs lesion raw_pearsonr: stat=8234.0, p-value=3.3912156126094973e-12
    Wilcoxon layer vs froi raw_pearsonr: stat=9593.0, p-value=1.6792161297103249e-22


Metric: raw_rdmr
  FROI: FFA
    Wilcoxon layer vs lesion raw_rdmr: stat=9864.0, p-value=5.754921264551868e-25
    Wilcoxon layer vs froi raw_rdmr: stat=9631.0, p-value=7.721292045609256e-23
  FROI: PPA
    Wilcoxon layer vs lesion raw_rdmr: stat=9398.0, p-value=8.209676073262376e-21
    Wilcoxon layer vs froi raw_rdmr: stat=9648.0, p-value=5.443405331580223e-23
  FROI: EBA
    Wilcoxon layer vs lesion raw_rdmr: stat=9146.0,

## Figure S10: L1-positive mapping

In [102]:
mapper = 'l1pos'
alpha = 0.01

em_csv_subset = em_csv[
    (em_csv['mapper'] == mapper) &
    (em_csv['alpha'] == alpha) &
    (em_csv['model_froi'].isin(core_frois)) &
    (em_csv['model_localizer'] == model_localizer) &
    (em_csv['model_tthreshold'] == tthreshold)
].copy()

# Estimate mean and median of difference between layer_raw_pearsonr and lesion_raw_pearsonr, as well as for rdmr
em_csv_subset['delta_pearsonr'] = (em_csv_subset['layer_raw_pearsonr'] - em_csv_subset['lesion_raw_pearsonr'])/em_csv_subset['fmri_splithalf_pearsonr']*100
em_csv_subset['delta_rdmr'] = (em_csv_subset['layer_raw_rdmr'] - em_csv_subset['lesion_raw_rdmr'])/em_csv_subset['fmri_splithalf_rdmr']*100
em_csv_subset_grouped = (
    em_csv_subset
    .groupby(['model', 'fmri_froi'])
    .agg(
        delta_pearsonr_lesion_median=('delta_pearsonr', 'median'),
        delta_pearsonr_lesion_mean=('delta_pearsonr', 'mean'),
        delta_rdmr_lesion_median=('delta_rdmr', 'median'),
        delta_rdmr_lesion_mean=('delta_rdmr', 'mean'),
    )
    .reset_index()
)
print('Delta pearsonr and rdmr between layer and lesion (grouped by ROI):')
display(em_csv_subset_grouped)
print(f'Median delta pearsonr (layer - lesion): {em_csv_subset_grouped["delta_pearsonr_lesion_median"].median()}')
print(f'Mean delta pearsonr (layer - lesion): {em_csv_subset_grouped["delta_pearsonr_lesion_mean"].mean()}')
print(f'Median delta rdmr (layer - lesion): {em_csv_subset_grouped["delta_rdmr_lesion_median"].median()}')
print(f'Mean delta rdmr (layer - lesion): {em_csv_subset_grouped["delta_rdmr_lesion_mean"].mean()}')
print('\n')

# Estimate the same between layer_raw_pearsonr and froi_raw_pearsonr, as well as for rdmr
em_csv_subset['delta_pearsonr_froi'] = (em_csv_subset['layer_raw_pearsonr'] - em_csv_subset['froi_raw_pearsonr'])/em_csv_subset['fmri_splithalf_pearsonr']*100
em_csv_subset['delta_rdmr_froi'] = (em_csv_subset['layer_raw_rdmr'] - em_csv_subset['froi_raw_rdmr'])/em_csv_subset['fmri_splithalf_rdmr']*100
em_csv_subset_grouped = (
    em_csv_subset
    .groupby(['model', 'fmri_froi'])
    .agg(
        delta_pearsonr_froi_median=('delta_pearsonr_froi', 'median'),
        delta_pearsonr_froi_mean=('delta_pearsonr_froi', 'mean'),
        delta_rdmr_froi_median=('delta_rdmr_froi', 'median'),
        delta_rdmr_froi_mean=('delta_rdmr_froi', 'mean'),
    )
    .reset_index()
)
print(f'Median delta pearsonr (layer vs froi): {em_csv_subset_grouped["delta_pearsonr_froi_median"].median()}')
print(f'Mean delta pearsonr (layer vs froi): {em_csv_subset_grouped["delta_pearsonr_froi_mean"].mean()}')
print(f'Median delta rdmr (layer vs froi): {em_csv_subset_grouped["delta_rdmr_froi_median"].median()}')
print(f'Mean delta rdmr (layer vs froi): {em_csv_subset_grouped["delta_rdmr_froi_mean"].mean()}')

test_normality = False

mapper = 'ridge'
alpha = 0.1

em_csv_subset = em_csv[
    (em_csv['mapper'] == mapper) &
    (em_csv['alpha'] == alpha) &
    (em_csv['model_froi'].isin(core_frois)) &
    (em_csv['model_localizer'] == model_localizer) &
    (em_csv['model_tthreshold'] == tthreshold)
]

em_metrics = ['raw_pearsonr', 'raw_rdmr']

for metric in em_metrics:
    print(f'Metric: {metric}')
    for froi in core_frois:
        froi_subset = em_csv_subset[
            (em_csv_subset['model_froi'] == froi) &
            (em_csv_subset['fmri_froi'] == froi)
        ]
        # Group by model and take median across subjects for each metric
        froi_subset_grouped = (
            froi_subset
            .groupby(['model'])
            .agg(
                **{
                    f'layer_{metric}': (f'layer_{metric}', 'median'),
                    f'lesion_{metric}': (f'lesion_{metric}', 'median'),
                    f'froi_{metric}': (f'froi_{metric}', 'median'),
                }
            )
            .reset_index()
        )
        assert len(froi_subset_grouped) == len(to_model_names), f'Expected {len(to_model_names)} models in the grouped subset for froi {froi}, but got {len(froi_subset_grouped)}'
        layer_metric = froi_subset_grouped[f'layer_{metric}']
        lesion_metric = froi_subset_grouped[f'lesion_{metric}']
        froi_metric = froi_subset_grouped[f'froi_{metric}']
        print(f'  FROI: {froi}')
        if test_normality:
            # Do a test of normality for each metric
            layer_normality_pval = stats.shapiro(layer_metric).pvalue
            lesion_normality_pval = stats.shapiro(lesion_metric).pvalue
            froi_normality_pval = stats.shapiro(froi_metric).pvalue
            # Print p-values and the interpretation
            print(f'    Layer {metric} normality p-value: {layer_normality_pval} -> {"normal" if layer_normality_pval >= 0.05 else "not normal"}')
            print(f'    Lesion {metric} normality p-value: {lesion_normality_pval} -> {"normal" if lesion_normality_pval >= 0.05 else "not normal"}')
            print(f'    FROI {metric} normality p-value: {froi_normality_pval} -> {"normal" if froi_normality_pval >= 0.05 else "not normal"}')
        # Do a paired Wilcoxon signed-rank test between layer and lesion metrics, and between layer and froi metrics
        wilcoxon_layer_lesion_stat, wilcoxon_layer_lesion_pval = stats.wilcoxon(layer_metric, lesion_metric, alternative='greater')
        wilcoxon_layer_froi_stat, wilcoxon_layer_froi_pval = stats.wilcoxon(layer_metric, froi_metric, alternative='greater')
        print(f'    Wilcoxon layer vs lesion {metric}: stat={wilcoxon_layer_lesion_stat}, p-value={wilcoxon_layer_lesion_pval}')
        print(f'    Wilcoxon layer vs froi {metric}: stat={wilcoxon_layer_froi_stat}, p-value={wilcoxon_layer_froi_pval}')
    print('\n')

Delta pearsonr and rdmr between layer and lesion (grouped by ROI):


,model,fmri_froi,delta_pearsonr_lesion_median,delta_pearsonr_lesion_mean,delta_rdmr_lesion_median,delta_rdmr_lesion_mean
0,alexnetbarlowtwins,EBA,0.305542,0.345859,0.239156,0.346639
1,alexnetbarlowtwins,FFA,3.689408,3.879838,4.554610,5.349533
2,alexnetbarlowtwins,PPA,0.762208,0.722076,1.618343,2.142731
3,alexnetimagenet1kv1,EBA,-0.130953,-0.109875,0.079597,0.044308
4,alexnetimagenet1kv1,FFA,6.331680,6.445305,7.624229,8.727008
...,...,...,...,...,...,...
100,vitl14kosmos2,FFA,4.471621,4.739674,10.732018,11.914604
101,vitl14kosmos2,PPA,0.626548,0.587202,1.253958,1.130612
102,vitl14openaiclip,EBA,-0.118398,-0.053182,-0.329491,-0.293773
103,vitl14openaiclip,FFA,8.347636,8.083584,18.438106,18.179857


Median delta pearsonr (layer - lesion): 0.6265478018148813
Mean delta pearsonr (layer - lesion): 1.3356646297854013
Median delta rdmr (layer - lesion): 1.246088216332006
Mean delta rdmr (layer - lesion): 2.565686359751006


Median delta pearsonr (layer vs froi): 3.75709732500802
Mean delta pearsonr (layer vs froi): 2.4060570489516713
Median delta rdmr (layer vs froi): 8.462148946530322
Mean delta rdmr (layer vs froi): 6.087507338671856
Metric: raw_pearsonr
  FROI: FFA
    Wilcoxon layer vs lesion raw_pearsonr: stat=630.0, p-value=2.9103830456733704e-11
    Wilcoxon layer vs froi raw_pearsonr: stat=625.0, p-value=2.9103830456733704e-10
  FROI: PPA
    Wilcoxon layer vs lesion raw_pearsonr: stat=611.0, p-value=8.934875950217247e-09
    Wilcoxon layer vs froi raw_pearsonr: stat=619.0, p-value=1.6007106751203537e-09
  FROI: EBA
    Wilcoxon layer vs lesion raw_pearsonr: stat=529.0, p-value=0.00012148515088483691
    Wilcoxon layer vs froi raw_pearsonr: stat=610.0, p-value=1.079752109944820